# Outcome construction and exploratory analysis

**Purpose.** This notebook documents the fixed-window outcome redesign and its descriptive evidence. The supervisor-confirmed update adds L and Z as outcome descriptors while keeping them outside every predictive model.

**Principal inputs.** Preserved first-spell files, the weekly chart panel, clean Y52/Y26 cohorts, fixed-window outcomes and `supervisor_confirmed_update_v1` descriptives.

**Main outputs.** Reconciled X/Y definitions, outcome distributions, horizon evidence, Y–L correlations and chart-presence fragmentation.


In [ ]:
from pathlib import Path
import json
import math
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

# Find the repository from the current working directory so the notebook
# remains portable when the project folder is moved or opened in Colab.
current = Path.cwd().resolve()
candidates = [current, *current.parents]
ROOT = next((p for p in candidates if (p / "scripts/thesis_utils_v2.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Run this notebook from within the Spotify_clv project.")
sys.path.insert(0, str(ROOT / "scripts"))

from thesis_utils_v2 import (
    SEED, assert_close, assert_unique_tracks, blocked_predictor_hits,
    concise_table, environment_table, extract_spotify_id, load_json,
    read_chart_archive, record_values, regression_metrics,
    save_and_show_figure, setup_notebook,
)

ROOT, FIGURE_DIR, RECORD_DIR = setup_notebook("02_outcome_construction_and_eda.ipynb")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
display(environment_table())
print(f"Random seed: {SEED}")


## From first exit to fixed-window chart presence

The earlier survival outcome measured the first contiguous chart spell. It remains useful for describing time to first exit, but it discards later chart presence from the primary outcome. This matters because 598 tracks have confirmed re-entry and the final chart histories contain 1,031 confirmed re-entry spells.


In [ ]:
survival = pd.read_csv(ROOT / "data/processed/chart_survival/first_spell_survival.csv")
reentry = pd.read_csv(ROOT / "data/processed/chart_survival/reentry_summary.csv")
strict = survival[survival.survival_outcome_eligible.eq(1)]
survival_summary = pd.DataFrame([
    ("all chart tracks", len(survival)),
    ("strict first-spell survival cohort", len(strict)),
    ("strict confirmed exits", int(strict.event_indicator.eq(1).sum())),
    ("strict right-censored", int(strict.right_censored.eq(1).sum())),
    ("confirmed re-entry songs", int(reentry.has_confirmed_reentry.sum())),
    ("confirmed re-entry spells", int(reentry.confirmed_reentry_count.sum())),
], columns=["quantity", "tracks or spells"])
display(survival_summary)
assert len(strict) == 3718
assert int(strict.event_indicator.sum()) + int(strict.right_censored.sum()) == len(strict)
assert int(reentry.has_confirmed_reentry.sum()) == 598
assert int(reentry.confirmed_reentry_count.sum()) == 1031


The professor-motivated redesign uses the first four **calendar** weeks as an early measurement period, not as a requirement that a song remain continuously present. For a confirmed entry at week 1:

- `X` is the number of chart appearances in weeks 1–4 and takes values 1–4;
- `Y_52` is the number of chart appearances in weeks 5–52, including re-entry, and is bounded 0–48;
- `Y_26` is the corresponding weeks 5–26 count, bounded 0–22;
- `L` records the last numbered week of presence, while `Z_count` records separate presence blocks. They are descriptors, not predictors of Y.

This definition retains songs that disappear during the early period and avoids the old four-week landmark filter, which selected songs that were already longer-lived.

## Direct reconstruction from the weekly panel

The next check reads the weekly state panel. Week indices 0–3 form X; indices 4–51 form Y_52. A chart state is counted only when it is explicitly `present`. Unknown source states are neither presence nor absence, and any track whose required window contains one was excluded before this stage.


In [ ]:
primary = pd.read_csv(ROOT / "data/processed/pre_model_analysis_ready_v1/analysis_ready_550k_52w.csv")
short = pd.read_csv(ROOT / "data/processed/pre_model_analysis_ready_v1/analysis_ready_550k_26w.csv")
assert_unique_tracks(primary, "clean 550k 52-week cohort")
assert_unique_tracks(short, "clean 550k 26-week cohort")

panel = pd.read_csv(
    ROOT / "data/processed/chart_survival/song_week_panel.csv",
    usecols=["spotify_track_id", "chart_week", "week_index_since_first_entry", "source_week_status", "chart_state"],
)
panel_primary = panel[panel.spotify_track_id.isin(primary.spotify_track_id)].copy()
required = panel_primary[panel_primary.week_index_since_first_entry.between(0, 51)]
unknown_in_required_window = required.source_week_status.eq("source_week_missing").sum()
assert unknown_in_required_window == 0

present = required.chart_state.eq("present")
reconstructed = required.assign(
    x_presence=(present & required.week_index_since_first_entry.between(0, 3)).astype(int),
    y_presence=(present & required.week_index_since_first_entry.between(4, 51)).astype(int),
).groupby("spotify_track_id", as_index=False).agg(
    X_reconstructed=("x_presence", "sum"),
    Y_52_reconstructed=("y_presence", "sum"),
)
check = primary[["spotify_track_id", "X", "Y_52"]].merge(
    reconstructed, on="spotify_track_id", validate="one_to_one"
)
assert (check.X == check.X_reconstructed).all()
assert (check.Y_52 == check.Y_52_reconstructed).all()
display(pd.DataFrame([
    ("tracks reconstructed", len(check)),
    ("X mismatches", int((check.X != check.X_reconstructed).sum())),
    ("Y_52 mismatches", int((check.Y_52 != check.Y_52_reconstructed).sum())),
    ("unknown states inside retained 52-week windows", int(unknown_in_required_window)),
], columns=["check", "value"]))


The direct panel reconstruction agrees for all 1,014 primary tracks. This is the central safeguard against treating a missing source week as a chart absence.

## Outcome distributions

Y is a bounded, highly dispersed count with a large point mass at zero. The distribution motivates the later comparison of count, bounded and two-part formulations; the presence of many zeros alone does not prove that a zero-inflated model is required.


In [ ]:
def outcome_summary(frame, outcome, upper):
    values = frame[outcome].astype(float)
    return {
        "outcome": outcome, "N": len(values), "mean": values.mean(),
        "median": values.median(), "standard deviation": values.std(ddof=1),
        "variance": values.var(ddof=1), "minimum": values.min(), "maximum": values.max(),
        "p25": values.quantile(.25), "p75": values.quantile(.75),
        "p90": values.quantile(.90), "p95": values.quantile(.95), "p99": values.quantile(.99),
        "zero percent": 100 * values.eq(0).mean(),
        "variance-to-mean": values.var(ddof=1) / values.mean(),
        "theoretical maximum": upper,
    }

summaries = pd.DataFrame([
    outcome_summary(primary, "Y_52", 48),
    outcome_summary(short, "Y_26", 22),
])
display(summaries.round(3))
assert primary.Y_52.between(0, 48).all() and primary.Y_52.max() == 48
assert short.Y_26.between(0, 22).all() and short.Y_26.max() == 22

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(primary.Y_52, bins=np.arange(-0.5, 49.5, 1), color="#4C78A8", edgecolor="white")
axes[0].set(title="Future chart presence over 52 weeks", xlabel="Y_52: chart weeks 5–52", ylabel="Tracks")
axes[1].hist(short.Y_26, bins=np.arange(-0.5, 23.5, 1), color="#72B7B2", edgecolor="white")
axes[1].set(title="Future chart presence over 26 weeks", xlabel="Y_26: chart weeks 5–26", ylabel="Tracks")
save_and_show_figure(fig, FIGURE_DIR / "nb02_outcome_distributions.png")


The mean exceeds the median substantially and the variance-to-mean ratio is well above one. More than half of each cohort has no chart appearance after week 4, while a smaller group remains present for many weeks.

## Early presence X and future longevity


In [ ]:
y_by_x = primary.groupby("X").Y_52.agg(
    N="size", mean_Y="mean", median_Y="median",
    p25=lambda x: x.quantile(.25), p75=lambda x: x.quantile(.75),
    zero_percent=lambda x: 100 * x.eq(0).mean(),
).reset_index()
display(y_by_x.round(3))

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.boxplot(data=primary, x="X", y="Y_52", showfliers=False, color="#72B7B2", ax=ax)
ax.set(title="Future chart weeks by early four-week presence", xlabel="X: appearances in calendar weeks 1–4", ylabel="Y_52: appearances in weeks 5–52")
save_and_show_figure(fig, FIGURE_DIR / "nb02_y52_by_x.png")


X=1–3 tracks usually have Y=0. The pronounced increase at X=4 explains why X is treated categorically and why it later dominates the predictive benchmarks. This is descriptive evidence, not a causal effect of remaining on the chart.

## What 52 weeks adds beyond 26 weeks


In [ ]:
same_tracks = primary[["spotify_track_id", "Y_52", "X"]].merge(
    short[["spotify_track_id", "Y_26"]], on="spotify_track_id", validate="one_to_one"
)
same_tracks["additional_weeks_27_to_52"] = same_tracks.Y_52 - same_tracks.Y_26
horizon_summary = pd.DataFrame([
    ("same tracks", len(same_tracks)),
    ("Pearson correlation", same_tracks[["Y_26", "Y_52"]].corr(method="pearson").iloc[0, 1]),
    ("Spearman correlation", same_tracks[["Y_26", "Y_52"]].corr(method="spearman").iloc[0, 1]),
    ("tracks with no additional weeks", same_tracks.additional_weeks_27_to_52.eq(0).sum()),
    ("tracks gaining any additional weeks", same_tracks.additional_weeks_27_to_52.gt(0).sum()),
    ("Y_26=0 but Y_52>0", ((same_tracks.Y_26 == 0) & (same_tracks.Y_52 > 0)).sum()),
    ("Y_26 at ceiling", same_tracks.Y_26.eq(22).sum()),
    ("Y_52 at ceiling", same_tracks.Y_52.eq(48).sum()),
], columns=["horizon comparison", "value"])
display(horizon_summary.round(4))
display(pd.read_csv(
    ROOT / "data/processed/final_pre_model_specification_v1/Y26_Y52_increment_distribution.csv"
))

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(same_tracks.Y_26, same_tracks.Y_52, alpha=.35, s=20, color="#4C78A8")
ax.plot([0, 22], [0, 22], linestyle="--", color="black", linewidth=1, label="No weeks added after week 26")
ax.set(xlabel="Y_26", ylabel="Y_52", title="Same-track comparison of the two horizons")
ax.legend(frameon=False)
save_and_show_figure(fig, FIGURE_DIR / "nb02_y26_y52_same_tracks.png")


Most tracks gain no extra chart week after week 26, but 139 do. The 52-week outcome therefore adds information mainly for persistent or later-returning songs. It also reduces later-year coverage: the shorter horizon retains 1,220 linked tracks and offers a much larger 2020 temporal sample. Y_52 remains the working primary target because it captures a full year of chart presence; Y_26 is retained as a sensitivity rather than treated as equivalent.

## Re-entry, calendar composition and chart threshold


In [ ]:
fixed = pd.read_csv(ROOT / "data/processed/fixed_window_count_v1/fixed_window_track_outcomes.csv")
fixed52 = fixed[(fixed.horizon_weeks.eq(52)) & fixed.fixed_window_eligible.eq(1)]
fixed_summary = pd.DataFrame([
    ("eligible chart tracks before feature matching", len(fixed52)),
    ("tracks with more than one presence block", int(fixed52.Z_reentry_more_than_one_block.sum())),
    ("correlation Y with L", fixed52[["Y_future_chart_weeks", "L_last_active_week_number"]].corr().iloc[0, 1]),
], columns=["diagnostic", "value"])
display(fixed_summary.round(3))

entry_composition = pd.read_csv(ROOT / "data/processed/pre_model_analysis_ready_v1/entry_year_composition.csv")
display(entry_composition[entry_composition.cohort.isin(["550k_52w", "550k_26w"])])

threshold = pd.read_csv(ROOT / "data/processed/fixed_window_count_v1/rank_200_annual_stream_summary.csv")
display(threshold.round(1))
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(threshold["year"], threshold["streams_mean"], marker="o", color="#E45756")
ax.set(title="Annual mean weekly streams at chart rank 200", xlabel="Chart year", ylabel="Weekly streams for rank-200 track")
ax.ticklabel_format(style="plain", axis="y")
save_and_show_figure(fig, FIGURE_DIR / "nb02_rank200_stream_threshold.png")

cohort_comparison = pd.read_csv(ROOT / "data/processed/pre_model_analysis_ready_v1/cohort_feature_comparison.csv")
display(cohort_comparison[["horizon_weeks", "feature_cohort", "tracks", "Y_mean", "Y_zero_percent", "main_difference"]].round(3))


## L and Z as descriptive outcome information

L is the last numbered week with chart presence inside the 52-week window. Y counts total future presence, whereas L records how far that presence extends. Z counts separate presence blocks in weeks 5–52; Z greater than one indicates fragmentation or re-entry. Both are calculated from the outcome window and are therefore descriptors, not admissible predictors.


In [ ]:
lz_summary = pd.read_csv(
    ROOT / "data/processed/supervisor_confirmed_update_v1/L_Z_descriptives.csv"
)
z_distribution = pd.read_csv(
    ROOT / "data/processed/supervisor_confirmed_update_v1/Z_distribution.csv"
)
pearson_y_l = primary[["Y_52", "L_52"]].corr(method="pearson").iloc[0, 1]
spearman_y_l = primary[["Y_52", "L_52"]].corr(method="spearman").iloc[0, 1]
expected = lz_summary.set_index("measure").value
assert_close(pearson_y_l, expected["Y52_L52_Pearson"], 1e-12, "Y52-L52 Pearson")
assert_close(spearman_y_l, expected["Y52_L52_Spearman"], 1e-12, "Y52-L52 Spearman")
assert np.array_equal(primary.Z_reentry_52.astype(int), primary.Z_count_52.gt(1).astype(int))
display(lz_summary.round(4))
display(z_distribution.round(3))

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(z_distribution.Z_count_52.astype(str), z_distribution.tracks, color="#4C78A8")
ax.set(title="Separate chart-presence blocks in weeks 5–52",
       xlabel="Z: number of presence blocks", ylabel="Tracks")
ax.spines[["top", "right"]].set_visible(False)
save_and_show_figure(fig, FIGURE_DIR / "nb02_z_presence_blocks.png")


Y and L are strongly related because songs with more observed chart weeks usually remain active farther into the horizon. The association is not perfect because Y allows gaps and re-entry. Most tracks have either no future block or one continuous future block; the smaller Z>1 group demonstrates why first exit is not always an absorbing endpoint.


In [ ]:
record_values(RECORD_DIR / "nb02_supervisor_update_values.json", "02_outcome_construction_and_eda", {
    "y52_l52_pearson": pearson_y_l,
    "y52_l52_spearman": spearman_y_l,
    "z_mean": expected["Z_mean"],
    "z_median": expected["Z_median"],
    "z_maximum": expected["Z_maximum"],
    "z_multiple_blocks_n": int(expected["Z_greater_than_1_count"]),
    "z_multiple_blocks_percent": expected["Z_greater_than_1_percent"],
})
print("Notebook 02 supervisor-update descriptive record saved.")


The chart threshold and exact-ID coverage change across calendar time. The 550k and 160k cohorts also differ in Y and X composition, so later common-feature comparisons are robustness checks rather than pooled analyses.

## Notebook 02 conclusions

The weekly panel reproduces X and Y_52 exactly for every primary track, with no unknown source state inside a retained observation window. Y_52 is a bounded count of chart presence after the early four-week measurement period and includes re-entry. Its strong nonlinear relationship with X is the principal descriptive finding carried into modelling. First-spell survival remains a secondary historical construct, while Y_26 tests sensitivity to a shorter horizon and improved later-year coverage.


In [ ]:
record_values(RECORD_DIR / "nb02_values.json", "02_outcome_construction_and_eda", {
    "primary_n": len(primary),
    "y52_min": primary.Y_52.min(),
    "y52_max": primary.Y_52.max(),
    "y52_mean": primary.Y_52.mean(),
    "y52_zero_percent": 100 * primary.Y_52.eq(0).mean(),
    "y26_n": len(short),
    "y26_max": short.Y_26.max(),
    "x4_n": int(primary.X.eq(4).sum()),
    "same_track_y26_y52_n": len(same_tracks),
    "tracks_with_post26_increment": int(same_tracks.additional_weeks_27_to_52.gt(0).sum()),
    "strict_survival_n": len(strict),
    "confirmed_reentry_songs": int(reentry.has_confirmed_reentry.sum()),
    "unknown_states_retained_window": int(unknown_in_required_window),
})
print("Notebook 02 reconciliation record saved.")
